# PSX Model Inference Demo
This notebook demonstrates how to load our trained Random Forest and Deep Learning (LSTM) models to predict the next day's stock price for PSO.
Since the models are already trained and saved as artifacts, we only need to pass them the most recent historical data.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import torch
import torch.nn as nn

# Add project root to sys path so we can import modules if needed
project_root = os.path.dirname(os.getcwd())
sys.path.append(project_root)


In [ ]:
# 1. Define LSTM Model Architecture
# PyTorch requires the class definition to be available when loading state dicts.
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, 
                            num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out


In [ ]:
# 2. Load Models and Scalers
MODELS_DIR = os.path.join(project_root, 'models')

# Load Random Forest
print("Loading Random Forest...")
rf_model = joblib.load(os.path.join(MODELS_DIR, 'baseline_rf_model.pkl'))

# Load Scalers
print("Loading Scalers...")
feature_scaler = joblib.load(os.path.join(MODELS_DIR, 'feature_scaler.pkl'))
target_scaler = joblib.load(os.path.join(MODELS_DIR, 'target_scaler.pkl'))

# Load LSTM
print("Loading Deep Learning LSTM...")
device = torch.device("cpu")
lstm_model = LSTMModel(input_dim=19) # 19 features
lstm_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'lstm_model.pth'), map_location=device))
lstm_model.eval() # Set model to evaluation mode!

print("All models loaded successfully! 🚀")


In [ ]:
# 3. Load Recent Historical Data
PROCESSED_DIR = os.path.join(project_root, 'data', 'processed')
df = pd.read_csv(os.path.join(PROCESSED_DIR, 'pso_features.csv'))

# Exclude metadata to get only the 19 predictive features
exclude_cols = ['ticker', 'date', 'created_at', 'target_close_t1']
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"Total historical records loaded: {len(df)}")
# Let's peek at the last 5 days
df[['date', 'close', 'volume', 'sma_7', 'rsi_14']].tail()


In [ ]:
# 4. Predict Tomorrow's Price!

# --- Random Forest Prediction ---
# Random forest only needs the very last row
latest_day_features = df[feature_cols].iloc[-1:]
rf_pred = rf_model.predict(latest_day_features)[0]

# --- LSTM Prediction ---
# LSTM requires a sequence of the last 30 days
lookback = 30
last_30_days = df[feature_cols].iloc[-lookback:].values

# 1. Scale the inputs using the exact same scaler from training
last_30_scaled = feature_scaler.transform(last_30_days)
# 2. Convert to PyTorch tensor and add a batch dimension (1, 30, 19)
tensor_x = torch.tensor(last_30_scaled, dtype=torch.float32).unsqueeze(0).to(device)

# 3. Forward pass
with torch.no_grad():
    lstm_pred_scaled = lstm_model(tensor_x)
    
# 4. Inverse transform the scaled prediction back to Rupees
lstm_pred = target_scaler.inverse_transform(lstm_pred_scaled.numpy())[0][0]

# Display Results
today_date = pd.to_datetime(df['date'].iloc[-1]).strftime('%Y-%m-%d')
today_close = df['close'].iloc[-1]

print(f"=====================================")
print(f"      PSX: PSO Predictions         ")
print(f"=====================================")
print(f"Last Trading Day ({today_date}): Rs. {today_close:.2f}")
print(f"-------------------------------------")
print(f"🤖 Random Forest (Tomorrow): Rs. {rf_pred:.2f}")
print(f"🧠 Deep Learning (Tomorrow): Rs. {lstm_pred:.2f}")
print(f"=====================================")


In [ ]:
# 5. Visualizing the Prediction
# Plot the last 60 days
historical = df.tail(60)

plt.figure(figsize=(12, 6))
plt.plot(pd.to_datetime(historical['date']), historical['close'], label='Historical Close Price', color='blue', marker='o', markersize=3)

# The 'x' coordinate for tomorrow
tomorrow_date = pd.to_datetime(historical['date'].iloc[-1]) + pd.Timedelta(days=1)

# Plot Predictions
plt.scatter(tomorrow_date, rf_pred, color='orange', s=100, label='RF Prediction', zorder=5)
plt.scatter(tomorrow_date, lstm_pred, color='green', s=100, label='LSTM Prediction', zorder=5)

plt.title('PSO - Next Day Price Prediction')
plt.xlabel('Date')
plt.ylabel('Price (Rs.)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
